# E4 EN chạy lại trên corpus cùng số token (T13 phần 2)

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

`E1_en` đã được thay bằng bản cắt đúng 38.250.964 token (phần 1), nhưng 6 run E4 EN
(`logspace`, `corpus`) vẫn train trên corpus cũ — mà dòng `uniform` của Bảng 7 lại
chính là `E1_en_HHHH`. Không chạy lại thì Bảng 4 và Bảng 7 vênh nhau cho cùng một
nhãn. Ở đây chạy lại 6 run E4 EN trên đúng corpus matched (cùng cache 110k bài,
cắt cùng đích); dòng `uniform` sau đó lấy từ E1_en matched.

Cấu hình giữ như E4 gốc: HHHH, nhánh corpus dùng alpha đo sẵn trong
`results/alpha_en_bpe500.json`. 6 run = logspace/corpus × seed 0–2, tốn ~12 phút
token hoá (phiên mới nên cache phải dựng lại) cộng ~1 giờ GPU T4, kết quả gói
trong `E4_en_matched_results.zip`.

Nhớ bật GPU trước: Runtime → Change runtime type → T4.


## 1 · Nạp mã nguồn

In [1]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK     = "/content/Hyena-Attention-Study"

import os, shutil, subprocess, sys

os.chdir("/content")             # thoát khỏi WORK trước khi xoá: rmtree thư mục
                                 # đang đứng làm git chết 128 "unable to read cwd"
if os.path.isdir(WORK):
    shutil.rmtree(WORK)          # chạy lại từ đầu -> luôn lấy bản mới nhất
subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

print("Đã clone vào", WORK)


Đã clone vào /content/Hyena-Attention-Study


## 2 · Cài thư viện và kiểm tra GPU

In [2]:
!pip install -q datasets tokenizers

import torch, datasets

print("torch", torch.__version__, "· datasets", datasets.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} · {p.total_memory/2**30:.1f} GB")
else:
    print("\n" + "!" * 70)
    print("!! CHƯA BẬT GPU — Runtime -> Change runtime type -> GPU (T4), rồi chạy lại")
    print("!" * 70)


torch 2.11.0+cu128 · datasets 4.0.0
GPU: Tesla T4 · 14.6 GB


## 3 · Kiểm trước khi đốt GPU

Ba bước, bước nào fail thì dừng: bản GitHub phải chứa E1_en matched (tức phần 1
đã tích hợp và push) cùng file alpha EN; test đường ống chạy CPU khoảng một phút;
dựng cache và xem tập train cắt ra đúng 38.250.964 chưa (~12 phút).


In [3]:
import json

d = json.load(open("results/E1_en_HHHH_s0.json"))
assert d["corpus"]["n_tokens_train"] == 38_250_964, (
    "results/E1_en_HHHH_s0.json trên GitHub chưa phải bản matched — "
    "push phần tích hợp T13 phần 1 lên main rồi chạy lại từ ô clone."
)
alpha = json.load(open("results/alpha_en_bpe500.json"))
assert len(alpha["alpha"]) == 256, "file alpha EN thiếu hoặc sai số kênh"
print("OK: E1_en matched đã có trên repo, alpha EN sẵn sàng.")


OK: E1_en matched đã có trên repo, alpha EN sẵn sàng.


In [4]:
!python tests/test_pipeline.py


  [PASS] P1a Token hoá âm tiết khứ hồi  (46)
  [PASS] P1b Chuẩn hoá NFC hợp nhất mã
  [PASS] P1c Số âm tiết đúng như mong đợi  (7)
  [PASS] P2a Đầu vào/nhãn lệch đúng 1  (9)
  [PASS] P2b Ngân sách token được tôn trọng  (5000)
  [PASS] P3a Hyena học được  (0.023)
  [PASS] P3b Transformer học được  (0.024)
  [PASS] P3c Mô hình lai học được  (0.023)
  [PASS] P3d Hyena học được KHÔNG cần pos-emb  (0.022)
  [PASS] P4a Đường ống alpha-corpus (E4)
  [PASS] P4b Lịch learning rate
  [PASS] P4c Cờ --max_train_tokens tồn tại
  [PASS] P4d Cắt corpus tách rời ngân sách bước

==> Toàn bộ 13 test ĐẠT


In [5]:
from hyena_study.data import cached_token_stream

train, val, test, tok, stats = cached_token_stream(
    lang="en", tokenizer="syllable", vocab_size=16000, n_docs=110000,
    data_seed=0, max_tokens=38250964, cache_root="data_cache",
)
print(f"train={len(train):,} val={len(val):,} test={len(test):,}")
assert len(train) == 38250964, f"train={len(train):,} != 38.250.964"
print("OK: cache đã dựng, tập train cắt đúng 38.250.964 token.")


[cache] khong co san, dang dung data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

[cache] da luu 43,319,051 token train
train=38,250,964 val=2,406,613 test=2,406,613
OK: cache đã dựng, tập train cắt đúng 38.250.964 token.


## 4 · Sáu lần chạy

Ba run `logspace` rồi ba run `corpus`, chạy tuần tự từng ô, ô nào lỗi chạy lại
riêng ô đó. Mỗi run để ý dòng `token: train=38,250,964`; riêng nhánh corpus phải
in `dùng alpha từ corpus: results/alpha_en_bpe500.json (256 kênh...)`.


In [6]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode logspace \
        --seed 0 --run_name E4_logspace_en_s0 --out_dir results_t13b


[E4_logspace_en_s0] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E4_logspace_en_s0] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E4_logspace_en_s0] alpha = log-spaced 1..512 token (doi chung cong bang)
[E4_logspace_en_s0] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E4_logspace_en_s0] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.9286 | |g| 2.05 | 0.8M token | 10s | 82.7k tok/s
  b   200/6103 | loss 6.2567 | |g| 0.82 | 1.6M token | 18s | 92.8k tok/s
  b   300/6103 | loss 5.8615 | |g| 0.72 | 2.5M token | 25s | 96.6k tok/s
  b   400/6103 | loss 5.5847 | |g| 0.96 | 3.3M token | 33s | 98.5k tok/s
  b   500/6103 | loss 5.4084 | |g| 0.92 | 4.1M token | 41s | 99.5k tok/s
  --> val loss 5.2170 | val PPL 184.37
  b   600/6103 | loss 5.2685 | |g| 1.16 | 4.9M token | 52s | 94.8k tok/s
  b   700

In [7]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode logspace \
        --seed 1 --run_name E4_logspace_en_s1 --out_dir results_t13b


[E4_logspace_en_s1] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E4_logspace_en_s1] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E4_logspace_en_s1] alpha = log-spaced 1..512 token (doi chung cong bang)
[E4_logspace_en_s1] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E4_logspace_en_s1] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.8379 | |g| 2.17 | 0.8M token | 10s | 85.1k tok/s
  b   200/6103 | loss 6.2648 | |g| 0.98 | 1.6M token | 18s | 91.8k tok/s
  b   300/6103 | loss 5.6513 | |g| 0.91 | 2.5M token | 26s | 94.0k tok/s
  b   400/6103 | loss 5.5092 | |g| 1.24 | 3.3M token | 34s | 95.1k tok/s
  b   500/6103 | loss 5.2710 | |g| 0.75 | 4.1M token | 43s | 95.8k tok/s
  --> val loss 5.2124 | val PPL 183.53
  b   600/6103 | loss 5.2084 | |g| 1.04 | 4.9M token | 54s | 91.4k tok/s
  b   700

In [8]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode logspace \
        --seed 2 --run_name E4_logspace_en_s2 --out_dir results_t13b


[E4_logspace_en_s2] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E4_logspace_en_s2] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E4_logspace_en_s2] alpha = log-spaced 1..512 token (doi chung cong bang)
[E4_logspace_en_s2] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E4_logspace_en_s2] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.8524 | |g| 2.29 | 0.8M token | 10s | 85.1k tok/s
  b   200/6103 | loss 6.1679 | |g| 1.47 | 1.6M token | 18s | 92.0k tok/s
  b   300/6103 | loss 5.8228 | |g| 1.06 | 2.5M token | 26s | 94.2k tok/s
  b   400/6103 | loss 5.4142 | |g| 0.91 | 3.3M token | 34s | 95.2k tok/s
  b   500/6103 | loss 5.2257 | |g| 1.08 | 4.1M token | 43s | 95.9k tok/s
  --> val loss 5.2152 | val PPL 184.04
  b   600/6103 | loss 5.1062 | |g| 0.87 | 4.9M token | 54s | 91.3k tok/s
  b   700

In [9]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode corpus --alpha_file results/alpha_en_bpe500.json \
        --seed 0 --run_name E4_corpus_en_s0 --out_dir results_t13b


[E4_corpus_en_s0] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E4_corpus_en_s0] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E4_corpus_en_s0] dùng alpha từ corpus: results/alpha_en_bpe500.json (256 kênh, nguồn: mutual_information_decay · lang=en · tokenizer=bpe)
[E4_corpus_en_s0] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E4_corpus_en_s0] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.9205 | |g| 2.03 | 0.8M token | 9s | 86.9k tok/s
  b   200/6103 | loss 6.2439 | |g| 0.96 | 1.6M token | 18s | 93.1k tok/s
  b   300/6103 | loss 5.8439 | |g| 0.71 | 2.5M token | 26s | 94.9k tok/s
  b   400/6103 | loss 5.5655 | |g| 0.69 | 3.3M token | 34s | 95.8k tok/s
  b   500/6103 | loss 5.3998 | |g| 0.89 | 4.1M token | 43s | 96.3k tok/s
  --> val loss 5.2085 | val PPL 182.83
  b   600/6103 | loss 5.25

In [10]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode corpus --alpha_file results/alpha_en_bpe500.json \
        --seed 1 --run_name E4_corpus_en_s1 --out_dir results_t13b


[E4_corpus_en_s1] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E4_corpus_en_s1] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E4_corpus_en_s1] dùng alpha từ corpus: results/alpha_en_bpe500.json (256 kênh, nguồn: mutual_information_decay · lang=en · tokenizer=bpe)
[E4_corpus_en_s1] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E4_corpus_en_s1] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.8479 | |g| 2.16 | 0.8M token | 9s | 89.7k tok/s
  b   200/6103 | loss 6.2571 | |g| 0.92 | 1.6M token | 17s | 94.6k tok/s
  b   300/6103 | loss 5.6411 | |g| 0.84 | 2.5M token | 26s | 96.0k tok/s
  b   400/6103 | loss 5.4921 | |g| 1.17 | 3.3M token | 34s | 96.7k tok/s
  b   500/6103 | loss 5.2635 | |g| 0.81 | 4.1M token | 42s | 97.1k tok/s
  --> val loss 5.2010 | val PPL 181.46
  b   600/6103 | loss 5.19

In [11]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode corpus --alpha_file results/alpha_en_bpe500.json \
        --seed 2 --run_name E4_corpus_en_s2 --out_dir results_t13b


[E4_corpus_en_s2] nạp corpus en (110000 bài) ...
[cache] DUNG LAI data_cache/en_syllable_v16000_d110000_s0_d24e0a4d3387  (43,319,051 token train)
[E4_corpus_en_s2] token: train=38,250,964 val=2,406,613 test=2,406,613 | vocab=16,000 | unk=0.1046 | ký tự/token=5.100
[E4_corpus_en_s2] dùng alpha từ corpus: results/alpha_en_bpe500.json (256 kênh, nguồn: mutual_information_decay · lang=en · tokenizer=bpe)
[E4_corpus_en_s2] tham số: 7,553,280 (mixer 1,219,328 · mlp 2,102,272 · emb 4,096,000)
[E4_corpus_en_s2] 6,103 bước × 8,192 token/bước = 49,995,776 token · warmup 305
  b   100/6103 | loss 7.8589 | |g| 2.29 | 0.8M token | 9s | 89.7k tok/s
  b   200/6103 | loss 6.1478 | |g| 1.18 | 1.6M token | 17s | 94.5k tok/s
  b   300/6103 | loss 5.7938 | |g| 1.06 | 2.5M token | 26s | 95.9k tok/s
  b   400/6103 | loss 5.3868 | |g| 0.94 | 3.3M token | 34s | 96.6k tok/s
  b   500/6103 | loss 5.2017 | |g| 0.94 | 4.1M token | 42s | 97.0k tok/s
  --> val loss 5.1950 | val PPL 180.37
  b   600/6103 | loss 5.08

## 5 · Kiểm và đóng gói

Mỗi file JSON phải ghi `corpus.n_tokens_train == 38.250.964` và `decay_mode`
khớp với `alpha_file`.


In [12]:
import json, glob

files = sorted(glob.glob("results_t13b/E4_*_en_s*.json"))
assert len(files) == 6, f"kỳ vọng 6 file JSON, thấy {len(files)}: {files}"
print(f"{'run':22s} {'decay':>9s} {'train tokens':>14s} {'test PPL':>9s}")
for f in files:
    d = json.load(open(f))
    n = d["corpus"]["n_tokens_train"]
    assert n == 38250964, f"{f}: n_tokens_train={n} != 38250964"
    mode = d["config"]["decay_mode"]
    assert (mode == "corpus") == bool(d["config"]["alpha_file"]), f"{f}: alpha_file lệch decay_mode"
    print(f"{d['run_name']:22s} {mode:>9s} {n:>14,d} {d['test_ppl']:>9.3f}")
print("\nOK: cả 6 run E4 EN đều trên corpus matched.")


run                        decay   train tokens  test PPL
E4_corpus_en_s0           corpus     38,250,964    55.080
E4_corpus_en_s1           corpus     38,250,964    54.781
E4_corpus_en_s2           corpus     38,250,964    54.879
E4_logspace_en_s0       logspace     38,250,964    55.106
E4_logspace_en_s1       logspace     38,250,964    54.789
E4_logspace_en_s2       logspace     38,250,964    55.003

OK: cả 6 run E4 EN đều trên corpus matched.


In [13]:
!zip -qr E4_en_matched_results.zip results_t13b
try:
    from google.colab import files
    files.download("E4_en_matched_results.zip")
except Exception as e:
    print("Không tự tải được (", e, ")")
    print("-> Tải tay: panel Files bên trái -> E4_en_matched_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>